Load data

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("../data/paysim_transactions.csv")
df.shape

(6362620, 11)

Dtypes

In [2]:
df.dtypes

step                int64
type               object
amount            float64
nameOrig           object
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest           object
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Null counts

In [3]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Class balance

In [4]:
fraud_counts = df["isFraud"].value_counts()
fraud_pct = df["isFraud"].mean() * 100
fraud_counts, fraud_pct

(isFraud
 0    6354407
 1       8213
 Name: count, dtype: int64,
 0.12908204481801522)

Transaction type distribution

In [5]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Summary statistics

In [6]:
df[["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]].describe()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06
std,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05
75%,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06
max,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08


nameOrig and nameDest format check

In [7]:
orig_prefixes = df["nameOrig"].str[0].value_counts()
dest_prefixes = df["nameDest"].str[0].value_counts()
orig_lengths = df["nameOrig"].str.len().value_counts()
dest_lengths = df["nameDest"].str.len().value_counts()
orig_prefixes, dest_prefixes, orig_lengths, dest_lengths

(nameOrig
 C    6362620
 Name: count, dtype: int64,
 nameDest
 C    4211125
 M    2151495
 Name: count, dtype: int64,
 nameOrig
 11    3398630
 10    2667372
 9      266807
 8       26789
 7        2718
 6         269
 5          35
 Name: count, dtype: int64,
 nameDest
 11    3397284
 10    2666336
 9      269498
 8       26411
 7        2736
 6         270
 5          43
 4          40
 2           2
 Name: count, dtype: int64)

Duplicate rows

In [8]:
df.duplicated().sum()

0

Fraud rate by transaction type

In [9]:
fraud_rate_by_type = df.groupby("type")["isFraud"].mean().reset_index()
fig = px.bar(fraud_rate_by_type, x="type", y="isFraud")
fig

Transaction amount distribution by isFraud

In [10]:
amounts = df["amount"].to_numpy()
is_fraud = df["isFraud"].to_numpy().astype(bool)

log_amount = np.log10(amounts + 1)
bins = np.linspace(log_amount.min(), log_amount.max(), 60)

counts_legit, edges = np.histogram(log_amount[~is_fraud], bins=bins)
counts_fraud, _ = np.histogram(log_amount[is_fraud], bins=bins)
bin_centers = (edges[:-1] + edges[1:]) / 2

hist_df = pd.DataFrame({
    "log_amount": np.concatenate([bin_centers, bin_centers]),
    "count": np.concatenate([counts_legit, counts_fraud]),
    "isFraud": [0] * len(bin_centers) + [1] * len(bin_centers),
})

fig = px.bar(hist_df, x="log_amount", y="count", color="isFraud", barmode="overlay")
fig

Transaction volume over step

In [11]:
volume_by_step = df.groupby("step").size().reset_index(name="count")
fig = px.line(volume_by_step, x="step", y="count")
fig